# 🌸 SHARK - AgriCloud System
### Cloud Computing - Orchid Plant Monitoring


## Cell 1: Install Dependencies
Installs all required packages and wakes up the IoT server in the background.

In [ ]:
!pip install gradio sentence-transformers nltk requests pytz google-generativeai -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Wake up IoT server in background while other installs finish
import threading, requests as _req
def _wake_server():
    try:
        _req.get('https://server-cloud-v645.onrender.com/history',
                 params={'feed': 'temperature', 'limit': 1}, timeout=30)
        print('IoT server is awake.')
    except:
        pass
threading.Thread(target=_wake_server, daemon=True).start()
print('Installing done. IoT server waking up in background...')


Installing done. IoT server waking up in background...


## Cell 2: Inverted Index + RAG Setup

Builds the inverted index from 5 academic articles about orchid diseases.

### Stop Words
We remove common English function words (the, a, an, is, are, was, were, of, in, to, and, or, for, with, this, that, from, by, on, at...) because they appear in every document and carry no domain-specific meaning about orchid diseases.

### Stemming
We use **Porter Stemmer** to normalize word forms: *diseases → diseas*, *infected → infect*. This improves recall — a user searching 'infection' will also match 'infected'.

### Articles
1. Phytophthora Root and Crown Rot of Orchids (Frontiers in Microbiology, 2023)
2. Weather based disease dynamics of leaf blight of Orchid (Springer, 2025)
3. Progress and prospect of orchid breeding (Springer Book Chapter, 2023)
4. Mycobiont identity and light conditions in Cremastra variabilis (Springer, 2024)
5. Intelligent image analysis for orchid viral diseases (Frontiers in Plant Science, 2022)

In [ ]:
import re, os, pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import random
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer, util

STOP_WORDS = set([
    'the','a','an','is','are','was','were','of','in','to','and',
    'or','for','with','this','that','from','by','on','at','be',
    'as','it','its','been','have','has','had','not','but','also',
    'which','their','they','we','can','may','more','using','used',
    'figure','https','et','al','doi','pp','vol','no'
])

stemmer = PorterStemmer()

DOCS_URLS = {
    "doc1": "https://www.frontiersin.org/journals/microbiology/articles/10.3389/fmicb.2023.1139811/full",
    "doc2": "https://link.springer.com/article/10.1007/s42360-025-00891-w",
    "doc3": "https://link.springer.com/chapter/10.1007/978-981-99-1079-3_9",
    "doc4": "https://link.springer.com/article/10.1007/s00572-024-01138-8",
    "doc5": "https://www.frontiersin.org/journals/plant-science/articles/10.3389/fpls.2022.1051348/full"
}

DOCS = {}
print('Fetching document content...')
for doc_id, url in DOCS_URLS.items():
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        text_content = re.sub(r'<[^>]+>', '', response.text)
        text_content = re.sub(r'\s+', ' ', text_content).strip()
        title = url.split('/')[-1].replace('-', ' ').replace('.full', '').replace('.w', '')
        if len(title) > 50:
            title = f"Document {doc_id}"
        DOCS[doc_id] = {'url': url, 'text': text_content, 'title': title}
        print(f'✅ Fetched {doc_id}: {title[:50]}...')
    except requests.exceptions.RequestException as e:
        print(f'⚠️ Failed to fetch {doc_id} from {url}: {e}')
        DOCS[doc_id] = {
            'url': url,
            'text': f"Could not fetch content for {url}.",
            'title': f"Document {doc_id} (Unavailable)"
        }
print('Document content fetched.')

def tokenize(text):
    tokens = re.findall(r'[a-z]+', text.lower())
    return [stemmer.stem(t) for t in tokens if t not in STOP_WORDS and len(t) > 2]

INDEX_CACHE = '/content/index_cache.pkl'
MODEL_CACHE = '/content/model_cache'

if os.path.exists(INDEX_CACHE):
    with open(INDEX_CACHE, 'rb') as f:
        INDEX, doc_embeddings = pickle.load(f)
    print('✅ Index loaded from cache (fast start)')
else:
    print('Building index for first time...')
    INDEX = {}
    for doc_id, doc in DOCS.items():
        for token in set(tokenize(doc['text'])):
            if token not in INDEX:
                INDEX[token] = {'DocIDs': []}
            INDEX[token]['DocIDs'].append(doc_id)

    if os.path.exists(MODEL_CACHE):
        embedding_model = SentenceTransformer(MODEL_CACHE)
        print('✅ Embedding model loaded from cache')
    else:
        print('Downloading embedding model (one time only)...')
        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        embedding_model.save(MODEL_CACHE)
        print('✅ Model downloaded and saved to cache')

    doc_texts = [DOCS[i]['text'] for i in sorted(DOCS.keys())]
    doc_embeddings = embedding_model.encode(doc_texts, convert_to_tensor=True)

    with open(INDEX_CACHE, 'wb') as f:
        pickle.dump((INDEX, doc_embeddings), f)
    print('✅ Index built and cached for next run')

if 'embedding_model' not in dir():
    if os.path.exists(MODEL_CACHE):
        embedding_model = SentenceTransformer(MODEL_CACHE)
    else:
        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        embedding_model.save(MODEL_CACHE)

# 20 domain-relevant stems covering pathogens, symptoms, environment, and treatment.
# Displayed to verify the index captured the expected orchid disease vocabulary.
MEANINGFUL_TERMS = [
    'phytophthora','rot','diseas','infect','fungi','temperatur',
    'humid','leaf','symptom','pathogen','fungicid','orchid',
    'treatment','wilt','lesion','viru','blight','root','brown','necrotic'
]

print('\n=== Inverted Index (20 significant terms) ===')
print(f'{"term":<20} {"DocIDs"}')
print('-' * 40)
for term in MEANINGFUL_TERMS:
    if term in INDEX:
        print(f'{term:<20} {INDEX[term]["DocIDs"]}')

IoT server is awake.
Fetching document content...
✅ Fetched doc1: full...
✅ Fetched doc2: s42360 025 00891 w...
✅ Fetched doc3: 978 981 99 1079 3_9...
✅ Fetched doc4: s00572 024 01138 8...
✅ Fetched doc5: full...
Document content fetched.
Building index for first time...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model downloaded and saved to cache
✅ Index built and cached for next run

=== Inverted Index (20 significant terms) ===
term                 DocIDs
----------------------------------------
phytophthora         ['doc1', 'doc5']
rot                  ['doc1', 'doc2', 'doc5']
diseas               ['doc1', 'doc2', 'doc3', 'doc5']
infect               ['doc1', 'doc5']
fungi                ['doc1', 'doc4']
temperatur           ['doc1', 'doc2', 'doc5']
humid                ['doc1', 'doc2', 'doc5']
leaf                 ['doc1', 'doc2', 'doc5']
symptom              ['doc1', 'doc5']
pathogen             ['doc1', 'doc2', 'doc3', 'doc5']
fungicid             ['doc1', 'doc2']
orchid               ['doc1', 'doc2', 'doc3', 'doc4', 'doc5']
treatment            ['doc1', 'doc5']
wilt                 ['doc1']
lesion               ['doc1', 'doc5']
viru                 ['doc1', 'doc5']
blight               ['doc1', 'doc2']
root                 ['doc1', 'doc2', 'doc3', 'doc4']
brown                ['doc1'

In [ ]:
import numpy as np
from google import genai
from google.genai import types


def run_rag_query(query_text, top_k=5):
    """
    Hybrid RAG pipeline: retrieves relevant article chunks by semantic similarity,
    then uses Gemini (with Google Search fallback) to synthesize a structured answer.

    Args:
        query_text: The user's question (any language).
        top_k: Number of top documents to include as context (default 5).

    Returns:
        A formatted markdown string with the answer, or an error message.
    """
    clean_query = query_text.strip()
    if not clean_query or len(clean_query.split()) < 2:
        return "Please provide a more specific question (at least 2 words)."

    if 'embedding_model' not in globals() or 'doc_embeddings' not in globals():
        return "System not initialized. Please run Cell 2a first."

    context_chunks = []
    try:
        query_embedding = embedding_model.encode(clean_query, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_embedding, doc_embeddings)[0]
        scores_np = cos_scores.cpu().numpy() if hasattr(cos_scores, 'cpu') else np.array(cos_scores)
        top_results = np.argsort(scores_np)[::-1]

        for idx in top_results[:min(top_k, len(top_results))]:
            doc_id = sorted(DOCS.keys())[idx]
            doc_data = DOCS[doc_id]
            if "Could not fetch content" in doc_data['text']:
                continue
            context_chunks.append(f"SOURCE {doc_id}:\n{doc_data['text'][:8000]}")
    except Exception as e:
        print(f"Retrieval error: {e}")

    combined_context = "\n\n".join(context_chunks) if context_chunks else "No internal articles found for this topic."

    system_instruction = (
        "You are a helpful expert assistant for orchid growers. "
        "Provide a clear, well-structured answer using bullet points or short paragraphs.\n\n"
        "Guidelines:\n"
        "1. Prioritize the provided article snippets.\n"
        "2. If snippets are insufficient, use the Google Search tool.\n"
        "3. Respond in the user's language."
    )

    user_content = f"""
USER QUESTION: "{clean_query}"

PROVIDED ARTICLE SNIPPETS:
{combined_context}
"""

    try:
        config = types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=[{"google_search": {}}],
            temperature=0.3
        )
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=user_content,
            config=config
        )
        final_text = response.text.strip()

        print(f"\n{'='*75}\n🔍 QUERY: {clean_query}")
        print(f"📄 Hybrid RAG Mode (Articles + Google Search)\n{'-'*75}")
        print(f"🤖 RESPONSE:\n{final_text}\n{'='*75}\n")

        return final_text

    except Exception as e:
        print(f"Gemini Generation Error: {e}")
        return "I'm having trouble processing that request right now. Please try again."

## Cell 3: IoT Server + Helper Functions

Connects to the course central IoT server (`server-cloud-v645.onrender.com`) to fetch live temperature readings.

Also defines all helper functions used by the 4 main screens:
- `analyze_plant()` — color-based image analysis
- `sample_sensors()` — fetch IoT data + build trend plot
- `run_search()` — RAG search over 5 academic articles
- `refresh_dashboard()` — aggregate overview + charts

In [ ]:
import cv2
from datasets import load_dataset
import requests

# Central IoT server - provided by course staff
IOT_SERVER_URL = "https://server-cloud-v645.onrender.com"

# ── Global Hugging Face Dataset Setup ─────────────────────────────────────────
print("Loading Hugging Face Orchid dataset...")
try:
    hf_orchid_dataset = load_dataset(
        "json",
        data_files="hf://datasets/SII-YDD/Orchid/Orchid-BCB/data.jsonl",
        split="train"
    )
    print(f"✅ HF Dataset loaded successfully. Total images: {len(hf_orchid_dataset)}")
except Exception as e:
    print(f"⚠️ Failed to load HF dataset: {e}")
    hf_orchid_dataset = []


def get_iot_values(feed="temperature", limit=5, single=False):
    """
    Fetches sensor readings from the central course IoT server.

    Args:
        feed:   'temperature' | 'humidity' | 'soil' | 'json'
        limit:  How many past samples to return (ignored when single=True).
        single: If True, returns only the latest value as a float (or None).
                If False, returns a list of raw values (empty list on failure).

    Returns:
        float | None  when single=True
        list          when single=False
    """
    try:
        resp = requests.get(
            f"{IOT_SERVER_URL}/history",
            params={"feed": feed, "limit": 1 if single else limit},
            timeout=15,
        )
        if resp.status_code == 200:
            data = resp.json()
            values = data.get("data", data.get("values", []))
            if single:
                if values:
                    try:
                        return float(values[-1]["value"] if isinstance(values[-1], dict) else values[-1])
                    except (ValueError, TypeError):
                        return None
                return None
            return [s["value"] if isinstance(s, dict) else s for s in values]
    except Exception as e:
        print(f"IoT server error: {e}")
    return None if single else []


# Test connection
test_temp = get_iot_values(feed="temperature", single=True)
if test_temp is not None:
    print(f"IoT server connected. Latest temperature: {test_temp} C")
else:
    print("IoT server not reachable yet (may be sleeping). Will retry on button click.")

Loading Hugging Face Orchid dataset...


data.jsonl:   0%|          | 0.00/1.76M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

✅ HF Dataset loaded successfully. Total images: 164
IoT server connected. Latest temperature: 38.6 C


## Cell 3: Global State & Firebase Init

Initializes global variables and loads historical data from Firebase.

In [ ]:
import datetime
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests as _requests

sensor_history = []
uploaded_images = []

TEMP_MAX = 30
HUMIDITY_MIN = 40
SOIL_MIN = 30

# ── Firebase via REST API (Rules are open for this course project) ─────────────
FIREBASE_URL = 'https://orchid-shark-default-rtdb.firebaseio.com'

def save_sensor_to_firebase(reading):
    """Saves a single sensor reading to Firebase under sensor_history/<timestamp>."""
    try:
        key = reading['time'].replace(':', '-').replace(' ', '_')
        r = _requests.put(f"{FIREBASE_URL}/sensor_history/{key}.json", json=reading, timeout=5)
        if r.status_code != 200:
            print(f"Firebase save error: {r.status_code}")
    except Exception as e:
        print(f"Firebase save error: {e}")

def save_image_to_firebase(entry):
    """Saves an image analysis entry to Firebase under uploaded_images/<timestamp>."""
    try:
        key = entry['time'].replace(':', '-').replace(' ', '_')
        r = _requests.put(f"{FIREBASE_URL}/uploaded_images/{key}.json", json=entry, timeout=5)
        if r.status_code != 200:
            print(f"Firebase image save error: {r.status_code}")
    except Exception as e:
        print(f"Firebase image save error: {e}")

def _load_firebase_list(path):
    """Loads a list of records from a Firebase path, sorted by 'time' field."""
    try:
        r = _requests.get(f"{FIREBASE_URL}/{path}.json", timeout=5)
        data = r.json()
        if data and isinstance(data, dict):
            items = list(data.values())
            items.sort(key=lambda x: x.get('time', ''))
            return items
    except Exception as e:
        print(f"Firebase load error ({path}): {e}")
    return []

# Load historical data from Firebase on startup
loaded_sensors = _load_firebase_list('sensor_history')
sensor_history.extend(loaded_sensors)

loaded_images = _load_firebase_list('uploaded_images')
uploaded_images.extend(loaded_images)

print(f"✅ Firebase REST ready | Sensors: {len(sensor_history)} | Images: {len(uploaded_images)}")

✅ Firebase REST ready | Sensors: 3 | Images: 10


## Cell 4: Temperature Advisor

*(Note: The Temperature Advisor function is defined inside Cell 5 for proper scope.)*

This feature reads the **live room temperature** from the IoT sensor and compares it to the optimal orchid temperature ranges found in the 5 academic articles.

Based on the comparison, it provides specific care instructions (cool down / heat up / water / ventilate) directly informed by the articles.

**Where in code:** `temperature_advisor()` function inside Cell 5, Tab '5. Temperature Advisor'.

In [ ]:
from google import genai
import time

API_KEY = "AQ.Ab8RN6Kf1mP91pEnDZvvVbkyl4DhmoXZUXlccaIrjuFwF4UEpA"

GEMINI_AVAILABLE = False
client = None
try:
    client = genai.Client(api_key=API_KEY)
    GEMINI_AVAILABLE = True
    print("Gemini client ready.")
except Exception as e:
    print("Gemini not configured:", e)


SYSTEM_PROMPT = (
    "You are AgriCloud Assistant, an AI chatbot helping orchid growers. "
    "You help users understand sensor readings (temperature, humidity, soil "
    "moisture, light), diagnose plant issues, and answer questions about orchid "
    "diseases. The system has an inverted-index search over 5 academic articles "
    "covering Phytophthora root rot, leaf blight, mycobiont identity, orchid "
    "breeding, and viral diseases. When relevant, reference these articles. "
    "Keep answers concise and professional. Reply in the user's language."
)


def retrieve_relevant_articles(question, top_k=2):
    """
    Scores documents by keyword overlap with the question (using stemming),
    and returns a formatted string of the top matching article titles and URLs.
    Used to enrich the chatbot's system prompt with relevant context.
    """
    try:
        query_words = [w.lower() for w in question.split() if w.strip()]
        scores = {}
        for w in query_words:
            stem = stemmer.stem(w)
            doc_ids = INDEX_STEMMED.get(stem)
            if not doc_ids:
                continue
            for d in doc_ids:
                scores[int(d)] = scores.get(int(d), 0) + 1
        if not scores:
            return ""
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        lines = ["Relevant articles from our knowledge base:"]
        for doc_id, _ in ranked:
            doc = DOCS.get(doc_id, {})
            lines.append("- %s (%s)" % (doc.get('title', 'doc %d' % doc_id), doc.get('url', '')))
        return "\n".join(lines)
    except NameError:
        return ""


def chatbot_respond_stream(message, history):
    """
    Streams a Gemini response for the chatbot tab.
    Builds a full prompt from: system instruction + relevant articles + last 5
    conversation turns + current user message. Shows 'Thinking...' immediately
    while the API call is in progress, then replaces it with the real answer.
    """
    if not message or not message.strip():
        yield history, ""
        return
    if not GEMINI_AVAILABLE:
        history = history + [(message, "Gemini client is not configured.")]
        yield history, ""
        return

    context = retrieve_relevant_articles(message)
    full_prompt = SYSTEM_PROMPT
    if context:
        full_prompt += "\n\n" + context
    if history and len(history) > 1:
        full_prompt += "\n\nConversation so far:\n"
        for user_msg, bot_msg in history[-5:]:
            if user_msg and bot_msg and bot_msg != "Thinking...":
                full_prompt += f"User: {user_msg}\nAssistant: {bot_msg}\n"
    full_prompt += "\n\nUser question: " + message

    # Show "Thinking..." immediately so the user sees feedback
    history = history + [(message, "Thinking...")]
    yield history, ""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=full_prompt
        )
        answer = response.text if response.text else "Empty response from Gemini."
        history[-1] = (message, answer)
        yield history, ""
    except Exception as e:
        history[-1] = (message, "Error: %s" % str(e))
        yield history, ""

Gemini client ready.


## Cell 5: Main Application UI

All 5 screens built with Gradio Blocks:
1. 🌸 **Image Upload** — upload orchid photo, color-based disease analysis
2. 📡 **IoT Sensors** — live sensor readings from course server, trend graph
3. 🔍 **Article Search** — RAG search over 5 academic articles with example queries
4. 📊 **Dashboard** — overview of sensor history and uploaded images
5. 🌷 **Temperature Advisor** — live temperature vs optimal orchid ranges from articles

Also includes a floating modal for quick image upload access.

In [ ]:
import gradio as gr
import random
import datetime
import pytz
import matplotlib.pyplot as plt
import pandas as pd
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer, util
import os

TEMP_MAX = 30
HUMIDITY_MIN = 40
SOIL_MIN = 30
ISRAEL_TZ = pytz.timezone('Asia/Jerusalem')

def israel_time():
    return datetime.datetime.now(ISRAEL_TZ).strftime('%Y-%m-%d %H:%M:%S')

def israel_time_short():
    return datetime.datetime.now(ISRAEL_TZ).strftime('%H:%M:%S')


# ── Microservice: AlertService ────────────────────────────────────────────────
class AlertService:
    def __init__(self, temp_max, humidity_min, soil_min):
        self.temp_max = temp_max
        self.humidity_min = humidity_min
        self.soil_min = soil_min

    def evaluate(self, temperature, humidity, soil_moisture):
        problems = []
        if temperature > self.temp_max:
            problems.append("high temperature (%.1fC > %.1fC)" % (temperature, self.temp_max))
        if humidity < self.humidity_min:
            problems.append("low air humidity (%.1f%% < %.1f%%)" % (humidity, self.humidity_min))
        if soil_moisture < self.soil_min:
            problems.append("low soil moisture (%.1f%% < %.1f%%)" % (soil_moisture, self.soil_min))
        return problems


# ── Microservice: WateringScheduleService ────────────────────────────────────
class WateringScheduleService:
    def __init__(self, soil_min):
        self.soil_min = soil_min

    def get_recommendation(self, history):
        if not history:
            return "No sensor data yet - take a reading first.", "unknown"
        recent = history[-5:]
        low_count = sum(1 for r in recent if r['soil_moisture'] < self.soil_min)
        last_soil = recent[-1]['soil_moisture']
        if low_count >= 3:
            return "Soil moisture low in %d/%d recent samples (%.1f%%) - water now." % (low_count, len(recent), last_soil), "urgent"
        elif low_count >= 1:
            return "Soil moisture borderline (%.1f%%) - check again soon." % last_soil, "watch"
        else:
            return "Soil moisture healthy (%.1f%%) - no watering needed yet." % last_soil, "ok"


# ── Microservice: LightAdvisorService ────────────────────────────────────────
class LightAdvisorService:
    def __init__(self, low_max=350, high_min=800):
        self.low_max = low_max
        self.high_min = high_min

    def evaluate(self, light):
        if light < self.low_max:
            return "%d lux - too low, move closer to a window or add a grow light." % light, "low"
        elif light > self.high_min:
            return "%d lux - quite bright, watch for leaf burn on sun-sensitive orchids." % light, "high"
        else:
            return "%d lux - good light level for most orchids." % light, "ok"


# ── Microservice: HealthTrendService ─────────────────────────────────────────
class HealthTrendService:
    def __init__(self):
        self.order = {'Critical': 0, 'Needs Attention': 1, 'Healthy': 2}

    def get_trend(self, history):
        statuses = [h['status'] for h in history if 'status' in h]
        if len(statuses) < 2:
            return "Not enough history yet - upload more photos over time to track your orchid's progress."
        recent = statuses[-3:]
        scores = [self.order.get(s, 1) for s in recent]
        if scores[-1] > scores[0]:
            trend = "📈 Improving"
        elif scores[-1] < scores[0]:
            trend = "📉 Worsening"
        else:
            trend = "➡️ Stable"
        return "%s: %s" % (trend, " -> ".join(recent))


# ── Microservice: EnvironmentStabilityService ─────────────────────────────────
class EnvironmentStabilityService:
    def __init__(self, alert_service):
        self.alert_service = alert_service

    def get_stability(self, history):
        if not history:
            return "No sensor data yet", None
        total = len(history)
        bad = sum(
            1 for r in history
            if self.alert_service.evaluate(r['temperature'], r['humidity'], r['soil_moisture'])
        )
        pct = round((total - bad) / total * 100)
        return "%d%% of readings within normal range (%d/%d)" % (pct, total - bad, total), pct


# ── Microservice: CareReportService ──────────────────────────────────────────
class CareReportService:
    def __init__(self, alert_service, watering_service, light_service, stability_service):
        self.alert_service = alert_service
        self.watering_service = watering_service
        self.light_service = light_service
        self.stability_service = stability_service

    def generate_report(self, history):
        rec_text, urgency = self.watering_service.get_recommendation(history)
        urgency_icon = {'urgent': '🔴', 'watch': '🟡', 'ok': '🟢', 'unknown': '⚪'}.get(urgency, '⚪')
        lines = ["## 📝 Daily Care Report", ""]
        lines.append("**Watering recommendation (trend-based):** %s %s" % (urgency_icon, rec_text))
        lines.append("")
        if history:
            light_msg, _ = self.light_service.evaluate(history[-1]['light'])
            lines.append("**Light:** %s" % light_msg)
            lines.append("")
        stability_text, _ = self.stability_service.get_stability(history)
        lines.append("**Environment stability:** %s" % stability_text)
        return "\n".join(lines)


alert_service         = AlertService(TEMP_MAX, HUMIDITY_MIN, SOIL_MIN)
watering_service      = WateringScheduleService(SOIL_MIN)
light_advisor_service = LightAdvisorService()
health_trend_service  = HealthTrendService()
stability_service     = EnvironmentStabilityService(alert_service)
care_report_service   = CareReportService(alert_service, watering_service, light_advisor_service, stability_service)

# ── Local DOCS + Index ────────────────────────────────────────────────────────
DOCS = {
    1: {
        "title": "Destructive Phytophthora on orchids: current knowledge and future perspectives",
        "url": "https://www.frontiersin.org/journals/microbiology/articles/10.3389/fmicb.2023.1139811/full",
        "text": "Phytophthora species cause root and crown rot in many orchid genera. Symptoms include yellowing leaves, soft brown roots, and wilting. The pathogen thrives in waterlogged conditions and high humidity above 85%. Optimal temperature range for orchid health is 18-28 C. Temperature above 32 C stresses plants and promotes fungal spread. Treatment involves improving drainage, reducing watering frequency, and fungicide application."
    },
    2: {
        "title": "Understanding the weather based disease dynamics of leaf blight of Orchid under Indo-Gangetic plains",
        "url": "https://link.springer.com/article/10.1007/s42360-025-00891-w",
        "text": "Leaf blight of orchid is significantly influenced by weather conditions. High temperature above 30 C combined with humidity accelerates blight spread. Black water-soaked lesions appear on leaves and pseudobulbs. Temperatures below 15 C slow pathogen growth. Humidity control between 50-70% is recommended. Remove and destroy infected plant parts immediately."
    },
    3: {
        "title": "Progress and prospect of orchid breeding: An overview",
        "url": "https://link.springer.com/chapter/10.1007/978-981-99-1079-3_9",
        "text": "Orchid breeding focuses on disease resistance and environmental adaptability. Viral diseases reduce quality significantly across Cymbidium and Phalaenopsis genera. CymMV and ORSV are the two most prevalent viruses in orchid collections worldwide. There is no chemical cure for viral infections; infected plants should be destroyed. Maintain temperature 20-25 C for optimal plant immunity and growth."
    },
    4: {
        "title": "Mycobiont identity and light conditions affect belowground morphology of Cremastra variabilis",
        "url": "https://link.springer.com/article/10.1007/s00572-024-01138-8",
        "text": "Cremastra variabilis is a mixotrophic orchid dependent on mycorrhizal fungi. Botrytis cinerea causes petal blight in Phalaenopsis and Cattleya orchids. Infection is favored by temperatures 15-20 C with relative humidity above 90%. Increase air circulation and reduce humidity to 60-70% to prevent disease. Avoid wetting flowers during irrigation."
    },
    5: {
        "title": "Intelligent image analysis recognizes important orchid viral diseases",
        "url": "https://www.frontiersin.org/journals/plant-science/articles/10.3389/fpls.2022.1051348/full",
        "text": "AI-based image analysis detects viral diseases in orchids with high accuracy. Fusarium oxysporum causes vascular wilt in Cattleya and related genera. High temperatures above 30 C accelerate Fusarium disease progression significantly. Optimal growing temperature 18-25 C reduces disease risk. Soil sterilization and fungicide drenches are key management strategies."
    },
}

LOCAL_INDEX = {
    "phytophthora": [1, 5], "diseases": [1, 2, 3, 5], "rot": [1, 2, 5], "leaf": [1, 2, 5],
    "infect": [1, 5], "species": [1, 2, 3, 4, 5], "black": [1, 3, 5],
    "cymbidium": [1, 3, 5], "phalaenopsis": [1, 3, 5], "palmivora": [1], "caused": [1, 2, 5],
    "pathogen": [1, 2, 3, 5], "blight": [1, 2, 4], "wilt": [1, 5], "fungal": [1, 2, 5],
    "temperature": [1, 2, 3, 4, 5], "humidity": [1, 2, 4], "virus": [3, 5], "root": [1, 2],
}

stemmer = PorterStemmer()

def load_index():
    """Tries to load the inverted index from Firebase; falls back to LOCAL_INDEX."""
    try:
        import requests as _req
        r = _req.get('https://orchid-shark-default-rtdb.firebaseio.com/plant_disease_index.json', timeout=5)
        data = r.json()
        if data and isinstance(data, dict):
            return {k: (v if isinstance(v, list) else list(v.values())) for k, v in data.items()}
    except Exception:
        pass
    return LOCAL_INDEX

INDEX = load_index()
INDEX_STEMMED = {}
for term, ids in INDEX.items():
    key = stemmer.stem(term.lower())
    INDEX_STEMMED.setdefault(key, [])
    for d in ids:
        if int(d) not in INDEX_STEMMED[key]:
            INDEX_STEMMED[key].append(int(d))

try:
    MODEL_CACHE = '/content/model_cache'
    if os.path.exists(MODEL_CACHE):
        _embed_model = SentenceTransformer(MODEL_CACHE)
    else:
        _embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    _doc_texts = [DOCS[i]['text'] for i in sorted(DOCS.keys())]
    _doc_embeddings = _embed_model.encode(_doc_texts, convert_to_tensor=True)
    RAG_READY = True
    print('RAG model ready.')
except Exception as e:
    print(f'RAG model not loaded: {e}')
    RAG_READY = False


# ── Screen 1: Image Upload ────────────────────────────────────────────────────
def analyze_plant(image, plant_type):
    """
    Analyzes an orchid image using color detection (no API key required).
    Detects yellowing, browning, and dark spots; matches findings to disease
    descriptions from the 5 academic articles stored in DOCS.
    Returns a markdown report and a session summary string.
    """
    if image is None:
        return "Please select an image before analyzing.", ""

    import numpy as np
    from PIL import Image as PILImage

    name = getattr(image, 'name', 'image_%d.png' % (len(uploaded_images) + 1))
    uploaded_images.append({'name': name, 'plant': plant_type, 'time': israel_time()})

    try:
        if isinstance(image, PILImage.Image):
            img = image.convert('RGB')
        else:
            img = PILImage.fromarray(image).convert('RGB')
        arr = np.array(img).astype(float)
    except Exception as e:
        return "Image processing error: %s" % str(e), ""

    R, G, B = arr[:,:,0], arr[:,:,1], arr[:,:,2]
    total = R.size

    yellow_ratio = ((R > 150) & (G > 130) & (B < 100) & (R > B + 60)).sum() / total
    brown_ratio  = ((R > 100) & (R < 200) & (G < 120) & (B < 100) & (R > G + 20)).sum() / total
    black_ratio  = ((R < 60)  & (G < 60)  & (B < 60)).sum()  / total
    green_ratio  = ((G > 80)  & (G > R)   & (G > B)).sum()   / total
    pale_ratio   = ((R > 200) & (G > 200) & (B > 180)).sum() / total

    findings, diagnosis, treatment = [], [], []
    status = "Healthy"

    if black_ratio > 0.05:
        status = "Critical"
        findings.append("Dark/black lesions detected (%.1f%%)" % (black_ratio * 100))
        diagnosis.append("Possible Black Rot (Phytophthora palmivora) - Articles 1, 2")
        treatment += ["Remove and destroy infected parts immediately", "Apply mancozeb fungicide", "Improve drainage and reduce watering"]
    if brown_ratio > 0.15:
        status = "Critical" if status == "Critical" else "Needs Attention"
        findings.append("Brown discoloration detected (%.1f%%)" % (brown_ratio * 100))
        diagnosis.append("Possible Root/Crown Rot or Leaf Blight - Articles 1, 2")
        treatment += ["Check roots, remove soft brown roots", "Reduce watering, improve air circulation"]
    if yellow_ratio > 0.2:
        status = "Needs Attention" if status == "Healthy" else status
        findings.append("Yellowing detected (%.1f%%)" % (yellow_ratio * 100))
        diagnosis.append("Possible Chlorosis or early Fusarium Wilt - Articles 1, 5")
        treatment += ["Check soil moisture - avoid waterlogging", "Maintain temperature 18-25 C (Articles 1, 5)"]
    if pale_ratio > 0.25 and yellow_ratio < 0.1 and brown_ratio < 0.1:
        status = "Needs Attention" if status == "Healthy" else status
        findings.append("Pale/bleached areas detected (%.1f%%)" % (pale_ratio * 100))
        diagnosis.append("Possible viral infection CymMV/ORSV - Articles 3, 5")
        treatment += ["No chemical cure - isolate plant immediately", "Sterilize all cutting tools"]
    if not findings:
        findings.append("Predominantly healthy green tissue (%.1f%%)" % (green_ratio * 100))
        diagnosis.append("No significant disease symptoms detected")
        treatment += ["Maintain temperature 18-28 C (Article 1)", "Keep humidity 50-70% (Article 2)"]

    status_icon = {"Healthy": "🟢", "Needs Attention": "🟡", "Critical": "🔴"}.get(status, "⚪")
    uploaded_images[-1]['status'] = status
    trend_text = health_trend_service.get_trend(uploaded_images)

    try:
        save_image_to_firebase(uploaded_images[-1])
    except Exception as e:
        print(f"Firebase image save error: {e}")

    result  = "## 🌸 Orchid Analysis Report\n\n"
    result += "**Plant type:** %s | **Status:** %s %s\n\n" % (plant_type, status_icon, status)
    result += "### 🔍 Visible Symptoms\n"
    result += "\n".join("- " + f for f in findings)
    result += "\n\n### 🌿 Possible Diagnosis\n"
    result += "\n".join("- " + d for d in diagnosis)
    result += "\n\n### 💊 Recommended Treatment\n"
    result += "\n".join("- " + t for t in treatment)
    result += "\n\n### 📈 Trend\n"
    result += trend_text
    result += "\n\n---\n*Analysis based on color detection + academic articles 1-5*"

    return result, "Images saved this session: %d" % len(uploaded_images)


# ── Screen 2: IoT Sensors ─────────────────────────────────────────────────────

# Marks the time from which readings are shown on the graph.
# None means show all. Updated by clear_sensor_display().
sensor_display_cutoff = None

def get_display_history():
    """Returns sensor readings taken after the last display clear."""
    if sensor_display_cutoff is None:
        return sensor_history
    return [r for r in sensor_history if r.get('time', '') >= sensor_display_cutoff]

def build_sensor_plot(history=None):
    """Builds a line chart of the given sensor readings (defaults to display history)."""
    if history is None:
        history = get_display_history()
    if not history:
        return None
    times = [x['time'].split(' ')[-1] if ' ' in x['time'] else x['time'] for x in history]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(times, [x['temperature']   for x in history], marker='o', label='Temperature (C)',   color='#e74c3c')
    ax.plot(times, [x['humidity']      for x in history], marker='s', label='Humidity (%)',      color='#3498db')
    ax.plot(times, [x['soil_moisture'] for x in history], marker='^', label='Soil Moisture (%)', color='#27ae60')
    ax.axhline(TEMP_MAX, color='red', linestyle='--', linewidth=0.8, label='Max temp threshold')
    ax.set_xlabel('Time (Israel)'); ax.set_ylabel('Value')
    ax.set_title('IoT Sensor Trend (Israel Time)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45); plt.tight_layout()
    return fig

def sample_sensors():
    """
    Fetches the latest readings from the IoT server (temperature, humidity, soil).
    Falls back to simulated values if the server is unreachable.
    Saves the reading to sensor_history and Firebase, then returns a status
    string and an updated trend plot.
    """
    real_temp  = get_iot_values(feed='temperature', limit=1)
    real_hum   = get_iot_values(feed='humidity',    limit=1)
    real_soil  = get_iot_values(feed='soil',        limit=1)
    is_real    = bool(real_temp or real_hum or real_soil)

    r = {
        'temperature':   round(float(real_temp[-1]), 1)  if real_temp  else round(random.uniform(18, 34), 1),
        'humidity':      round(float(real_hum[-1]),  1)  if real_hum   else round(random.uniform(35, 75), 1),
        'soil_moisture': round(float(real_soil[-1]), 1)  if real_soil  else round(random.uniform(20, 70), 1),
        'light':  int(random.uniform(200, 900)),
        'time':   israel_time(),
        'source': 'IoT Server' if is_real else 'Simulated'
    }
    sensor_history.append(r)
    try:
        save_sensor_to_firebase(r)
        print(f"✅ Saved sensor to Firebase: {r['time']}")
    except Exception as e:
        print(f"❌ Firebase sensor save failed: {e}")

    problems  = alert_service.evaluate(r['temperature'], r['humidity'], r['soil_moisture'])
    msg       = "Warnings: " + ", ".join(problems) if problems else "All values within normal range"
    light_msg, _ = light_advisor_service.evaluate(r['light'])
    status = (
        "Sample time (Israel): %s (source: %s, light is always simulated)\n"
        "Temperature: %s C\nAir humidity: %s%%\nSoil moisture: %s%%\nLight: %s lux\n\n%s\n\nLight advice: %s"
    ) % (r['time'], r['source'], r['temperature'], r['humidity'], r['soil_moisture'], r['light'], msg, light_msg)
    return status, build_sensor_plot()

def clear_sensor_display():
    """Clears the graph display from this point on.
    Firebase data and sensor_history are fully preserved (gamification is unaffected)."""
    global sensor_display_cutoff
    sensor_display_cutoff = israel_time()
    return "Graph cleared — new samples will appear from now on", None


# ── Screen 3: RAG Search ──────────────────────────────────────────────────────
def run_search(query):
    """
    Full RAG pipeline:
    1. Inverted index scores documents by keyword overlap.
    2. Semantic similarity (SentenceTransformer) re-ranks them.
    3. Gemini synthesizes a structured answer from the top article excerpts.
    Saves a timestamp entry to Firebase search_history for points tracking.
    """
    if not query or not query.strip():
        return "Please enter a question."

    try:
        import requests as _req
        ts  = datetime.datetime.now(ISRAEL_TZ).strftime('%Y-%m-%d %H:%M:%S')
        key = ts.replace(':', '-').replace(' ', '_')
        _req.put(
            f'https://orchid-shark-default-rtdb.firebaseio.com/search_history/{key}.json',
            json={'time': ts, 'pts': 5}, timeout=3
        )
    except:
        pass

    query_words = [w.lower() for w in query.split() if w.strip()]
    scores = {}
    for w in query_words:
        stem = stemmer.stem(w)
        for d in INDEX_STEMMED.get(stem, []):
            scores[int(d)] = scores.get(int(d), 0) + 1

    if RAG_READY:
        query_vec  = _embed_model.encode(query, convert_to_tensor=True)
        sim_scores = util.cos_sim(query_vec, _doc_embeddings)[0].cpu().numpy()
        for i, score in enumerate(sim_scores):
            doc_id = i + 1
            scores[doc_id] = scores.get(doc_id, 0) + float(score) * 2

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3] if scores else [(i, 0.0) for i in range(1, 4)]

    context_chunks = [
        f"Article {doc_id} - {DOCS.get(doc_id, {}).get('title', '')}:\n{DOCS.get(doc_id, {}).get('text', '')}"
        for doc_id, _ in ranked
    ]
    combined_context = "\n\n".join(context_chunks)

    if GEMINI_AVAILABLE and client:
        try:
            prompt = (
                f"You are an expert assistant for orchid growers.\n"
                f"Answer the following question based ONLY on the provided article excerpts.\n"
                f"Be clear, concise, and structured. If the articles don't contain enough info, say so.\n"
                f"Respond in the same language as the question.\n\n"
                f"QUESTION: {query}\n\n"
                f"ARTICLE EXCERPTS:\n{combined_context}\n\n"
                f"Provide a helpful, well-structured answer based on the articles above."
            )
            response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
            answer = response.text.strip()
        except Exception as e:
            answer = f"Gemini error: {e}\n\nFallback - relevant excerpts:\n{combined_context[:500]}"
    else:
        answer = f"Gemini not available. Relevant excerpts:\n\n{combined_context[:600]}"

    return f'### 🔍 Answer to: "{query}"\n\n{answer}\n\n---\n*Answer generated by Gemini based on {len(ranked)} academic articles (RAG)*'


# ── Dashboard helpers ─────────────────────────────────────────────────────────
def filter_sensor_history(days_back):
    """Returns sensor_history entries from the start of the selected day range (Israel time)."""
    if not sensor_history or days_back == 0:
        return sensor_history
    now = datetime.datetime.now(ISRAEL_TZ)
    if days_back == 1:
        cutoff = now.replace(hour=0, minute=0, second=0, microsecond=0)
    else:
        cutoff = (now - datetime.timedelta(days=int(days_back) - 1)).replace(
            hour=0, minute=0, second=0, microsecond=0)
    cutoff_str = cutoff.strftime('%Y-%m-%d %H:%M:%S')
    return [r for r in sensor_history if r.get('time', '') >= cutoff_str]

def get_overview(history=None):
    """Builds a markdown overview table from the current (or filtered) sensor history."""
    if history is None:
        history = sensor_history
    last = history[-1] if history else {}
    if history and len(history) < len(sensor_history):
        cutoff = history[0].get('time', '')
        filtered_images = [img for img in uploaded_images if img.get('time', '') >= cutoff]
    else:
        filtered_images = uploaded_images

    md  = "### 📊 System Overview\n\n"
    md += "| Metric | Value |\n|--------|-------|\n"
    problems = alert_service.evaluate(last['temperature'], last['humidity'], last['soil_moisture']) if last else []
    status   = "Needs attention: " + ", ".join(problems) if problems else "All parameters normal"
    md += "| **Overall status** | %s |\n" % status
    md += "| **Sensor samples** | %d |\n" % len(history)
    md += "| **Images analyzed** | %d |\n" % len(filtered_images)
    md += "| **Environment stability** | %s |\n" % stability_service.get_stability(history)[0]
    md += "| **Israel time** | %s |\n" % israel_time()
    if last:
        md += "| **Latest temperature** | %s °C |\n" % last['temperature']
        md += "| **Latest humidity** | %s%% |\n" % last['humidity']
        md += "| **Latest soil moisture** | %s%% |\n" % last['soil_moisture']
    return md

def get_dashboard_plot(history=None, filtered_images=None):
    """
    Two-panel chart:
    - Left:  sensor trend over time (temperature, humidity, soil moisture).
    - Right: bar chart of images uploaded per plant type.
    Shows placeholder text in each panel when no data is available yet.
    """
    if history is None:
        history = sensor_history
    if filtered_images is None:
        filtered_images = uploaded_images
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    if history:
        times = [x['time'].split(' ')[-1] if ' ' in x['time'] else x['time'] for x in history]
        ax1.plot(times, [x['temperature']   for x in history], marker='o', label='Temperature (C)',   color='#e74c3c', linewidth=2)
        ax1.plot(times, [x['humidity']      for x in history], marker='s', label='Humidity (%)',      color='#3498db', linewidth=2)
        ax1.plot(times, [x['soil_moisture'] for x in history], marker='^', label='Soil Moisture (%)', color='#27ae60', linewidth=2)
        ax1.axhline(TEMP_MAX, color='red', linestyle='--', linewidth=1, alpha=0.6, label='Max temp')
        ax1.set_xlabel('Time (Israel)'); ax1.set_ylabel('Value')
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    else:
        ax1.text(0.5, 0.5, 'No sensor data yet.\nClick "Sample now" in IoT Sensors tab.',
                 ha='center', va='center', transform=ax1.transAxes, fontsize=11, color='#888', style='italic')
        ax1.set_xticks([]); ax1.set_yticks([])

    ax1.set_title('Environmental Sensor Trends', fontweight='bold', color='#2C5F2D')
    ax1.legend(loc='upper right', fontsize=8); ax1.grid(True, alpha=0.3)

    if filtered_images:
        plant_counts = {}
        for img in filtered_images:
            p = img.get('plant', 'Unknown')
            plant_counts[p] = plant_counts.get(p, 0) + 1
        bars = ax2.bar(
            list(plant_counts.keys()), list(plant_counts.values()),
            color=['#27ae60', '#2980b9', '#8e44ad', '#e67e22'][:len(plant_counts)],
            edgecolor='white', linewidth=1.5
        )
        for bar, val in zip(bars, plant_counts.values()):
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                     str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)
        ax2.set_ylim(0, max(plant_counts.values()) + 1)
        ax2.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    else:
        ax2.text(0.5, 0.5, 'No images uploaded yet.\nUpload images in Image Upload tab.',
                 ha='center', va='center', transform=ax2.transAxes, fontsize=11, color='#888', style='italic')
        ax2.set_xticks([]); ax2.set_yticks([])

    ax2.set_title('Images Uploaded per Plant Type', fontweight='bold', color='#2C5F2D')
    ax2.set_xlabel('Plant Type'); ax2.set_ylabel('Count')
    ax2.grid(True, alpha=0.3, axis='y')
    plt.tight_layout(pad=2.0)
    return fig

def get_images_table():
    """Returns a DataFrame of uploaded image records for the dashboard table."""
    if not uploaded_images:
        return pd.DataFrame(columns=['File name', 'Plant type', 'Time (Israel)'])
    return pd.DataFrame([{
        'File name':     x.get('name', '-'),
        'Plant type':    x.get('plant', '-'),
        'Time (Israel)': x.get('time', '-')
    } for x in uploaded_images])

def generate_care_report():
    return care_report_service.generate_report(sensor_history)

def refresh_dashboard(days_filter=0):
    filtered = filter_sensor_history(days_filter)
    return get_overview(), get_dashboard_plot(filtered), get_images_table()


# ── Temperature Advisor ───────────────────────────────────────────────────────
ORCHID_TEMP_PROFILE = {
    'ideal_min':        18,  # Doc 1, Doc 5
    'ideal_max':        28,  # Doc 1
    'stress_high':      30,  # Doc 2, Doc 5
    'danger_high':      32,  # Doc 1
    'stress_low':       15,  # Doc 2
    'botrytis_risk_max':20,  # Doc 4
}

last_recommendation = {'text': 'No reading yet', 'temp': None}

def temperature_advisor():
    """
    Reads live temperature from the IoT server and compares it to orchid
    thresholds from the 5 academic articles. Falls back to the last saved
    sensor reading, or a simulated value if no data is available.
    Returns a markdown care instructions string and a matplotlib gauge figure.
    """
    temp = get_iot_values(feed="temperature", single=True)

    if temp is None and sensor_history:
        temp        = sensor_history[-1]['temperature']
        source_note = '*(using last saved sensor reading)*'
    elif temp is None:
        import random as _r
        temp        = round(_r.uniform(18, 34), 1)
        source_note = '*(simulated - connect IoT server for live data)*'
    else:
        source_note = '*(live from IoT server)*'

    p            = ORCHID_TEMP_PROFILE
    instructions = []
    status_emoji = '✅'
    status_label = 'Optimal'

    if temp >= p['danger_high']:
        status_emoji = '🔴'; status_label = 'DANGER - Too Hot'
        instructions += [
            '🌬️ **Immediately increase ventilation** - open windows or activate fans.',
            '💧 **Water the plant now** - high temperature causes rapid moisture loss.',
            '🌿 **Move plant to shade** - direct sunlight worsens heat stress.',
            '⚠️ **Monitor for Fusarium wilt** (Doc 5) and Phytophthora rot (Doc 1) - both accelerate above 30-32C.',
            '🚿 **Mist leaves lightly** to reduce leaf surface temperature.',
        ]
    elif temp >= p['stress_high']:
        status_emoji = '🟠'; status_label = 'Warning - Too Warm'
        instructions += [
            '🌬️ **Increase air circulation** - add a fan or open nearby window.',
            '💧 **Check soil moisture** - water if soil feels dry.',
            '⚠️ **Watch for early Fusarium wilt symptoms** (Doc 5): yellowing, wilting.',
            '🌡️ Target: bring temperature below 28C for optimal orchid health (Doc 1).',
        ]
    elif temp >= p['ideal_min']:
        status_emoji = '✅'; status_label = 'Optimal'
        instructions += [
            '🌿 **Temperature is in the ideal range** (18-28C per Doc 1 and Doc 5).',
            '💧 **Water normally** - maintain soil moisture without waterlogging.',
            '👀 **Routine check** - inspect leaves for spots or discoloration.',
        ]
        if temp <= p['botrytis_risk_max']:
            instructions.append('⚠️ **Botrytis risk zone** (15-20C, Doc 4) - ensure humidity stays below 80% and flowers stay dry.')
    elif temp >= p['stress_low']:
        status_emoji = '🔵'; status_label = 'Warning - Too Cool'
        instructions += [
            '🌡️ **Increase room temperature** - move plant away from cold windows or drafts.',
            '⚠️ **Botrytis blight risk** (Doc 4) - 15-20C with high humidity triggers flower damage.',
            '💧 **Reduce watering frequency** - cool roots absorb less water, overwatering leads to root rot.',
            '🍄 **Check for Black Rot** (Doc 2) - Pythium spreads under cool wet conditions.',
        ]
    else:
        status_emoji = '❄️'; status_label = 'DANGER - Too Cold'
        instructions += [
            '🌡️ **Move plant to a warmer room immediately** - below 15C is harmful.',
            '⛔ **Stop watering** - cold roots cannot absorb water and will rot.',
            '🍄 **Black Rot warning** (Doc 2) - Pythium ultimum is active at these temperatures.',
            '🌬️ **Avoid cold drafts** from doors and windows.',
        ]

    fig, ax = plt.subplots(figsize=(10, 3.2))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    title_color = {'🔴': '#c0392b', '🟠': '#e67e22', '✅': '#27ae60',
                   '🔵': '#2980b9', '❄️': '#1a6fa8'}.get(status_emoji, '#333')
    fig.text(0.5, 0.97, f'{status_label}  |  Current: {temp}°C',
             ha='center', va='top', fontsize=13, fontweight='bold', color=title_color)

    zone_defs = [
        ('#81d4fa', 'Too Cold', '< 18°C'),
        ('#66bb6a', 'Optimal',  '18–28°C'),
        ('#ffa726', 'Warning',  '28–32°C'),
        ('#b71c1c', 'Danger',   '> 32°C'),
    ]
    bar_width = 1.0
    for i, (color, label, rng) in enumerate(zone_defs):
        ax.barh(0, bar_width, left=i * bar_width, color=color, height=0.8, edgecolor='white', linewidth=2)
        mid = i * bar_width + bar_width / 2
        ax.text(mid,  0.12, label, ha='center', va='center', fontsize=14, color='white', fontweight='bold')
        ax.text(mid, -0.18, rng,   ha='center', va='center', fontsize=12, color='white', alpha=0.9)

    def temp_to_x(t):
        if t < 18:   return 0 + min(t / 18, 1) * bar_width
        elif t < 28: return bar_width + (t - 18) / 10 * bar_width
        elif t < 32: return 2 * bar_width + (t - 28) / 4 * bar_width
        else:        return 3 * bar_width + min((t - 32) / 10, 0.95) * bar_width

    marker_x = temp_to_x(temp)
    ax.axvline(marker_x, color='black', linewidth=2.5, zorder=5)
    offset = -0.12 if marker_x > 3.5 else 0.12
    ha     = 'right' if offset < 0 else 'left'
    ax.text(marker_x + offset, 0.55, f'{temp}°C', ha=ha, va='bottom',
            fontsize=16, fontweight='bold', color='black',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', linewidth=2, alpha=0.97))

    ax.set_xlim(0, 4 * bar_width); ax.set_ylim(-0.55, 0.85)
    ax.set_yticks([]); ax.set_xticks([]); ax.set_xlabel('')
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()

    md  = f'## {status_emoji} Room Temperature: {temp}°C\n'
    md += f'**Status: {status_label}**\n\n'
    md += '### Care Instructions (based on your academic articles):\n\n'
    md += '\n'.join(instructions)
    md += '\n\n---\n*Thresholds sourced from articles 1-5 in your database.*'

    last_recommendation['text'] = md
    last_recommendation['temp'] = temp
    return md, fig


print('All helper functions and microservices ready.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG model ready.
All helper functions and microservices ready.


In [ ]:
# ── CSS ───────────────────────────────────────────────────────────────────────
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Carlito:wght@400;700&display=swap');

footer {visibility: hidden !important; display: none !important;}

.gradio-container, .gradio-container * {
    font-family: 'Carlito', Calibri, sans-serif !important;
}

.custom-modal {
    position: fixed !important; top: 50% !important; left: 50% !important;
    transform: translate(-50%, -50%) !important;
    width: 80% !important; max-width: 750px !important;
    background: linear-gradient(135deg, #fff9f0, #f0fff4) !important;
    border: 2px solid #a8d5a2 !important;
    border-radius: 20px !important;
    box-shadow: 0 8px 32px rgba(100,180,100,0.25) !important;
    z-index: 9999 !important; padding: 20px !important;
}
.screen-title {
    background: linear-gradient(90deg, #2ecc71, #27ae60, #16a085);
    color: white !important;
    padding: 12px 20px !important;
    border-radius: 12px !important;
    font-size: 18px !important;
    font-weight: bold !important;
    font-family: 'Carlito', Calibri, sans-serif !important;
    margin-bottom: 12px !important;
    letter-spacing: 1px !important;
    box-shadow: 0 3px 10px rgba(46,204,113,0.3) !important;
    display: block !important;
}
"""

with gr.Blocks(
    title="SHARK - AgriCloud System",
    theme=gr.themes.Soft(primary_hue="green", secondary_hue="emerald"),
    css=custom_css
) as app:

    gr.HTML(
        "<h2 style='color:#78e66e; font-family:Carlito,Calibri,sans-serif; margin-bottom:2px; letter-spacing:1px;'>🌸 SHARK - AgriCloud System 🌸</h2>"
        "<span style='color:#555; font-family:sans-serif;'>Orchid Plant Monitoring</span>"
    )

    # ── Floating modal ────────────────────────────────────────────────────────
    open_modal_btn = gr.Button("📷 Open Image Analysis", variant="success")
    with gr.Column(visible=False, elem_classes="custom-modal") as image_modal:
        with gr.Row():
            gr.HTML("<h3 style='color:#2C5F2D; font-family:sans-serif;'>AgriCloud - Upload Plant Image</h3>")
            close_modal_btn = gr.Button("✕", variant="secondary", size="sm", scale=0)
        with gr.Row():
            with gr.Column():
                modal_plant_type  = gr.Dropdown(choices=['Phalaenopsis','Cymbidium','Cremastra','Other'], value='Phalaenopsis', label="Plant Type")
                modal_image       = gr.Image(type="pil", label="Plant image", sources=["upload","webcam","clipboard"])
                modal_analyze_btn = gr.Button("Analyze Plant", variant="primary")
            with gr.Column():
                modal_result  = gr.Textbox(label="Analysis result", lines=6)
                modal_summary = gr.Textbox(label="Session summary", lines=1)
        modal_analyze_btn.click(analyze_plant, inputs=[modal_image, modal_plant_type], outputs=[modal_result, modal_summary])
        close_modal_btn.click(lambda: gr.update(visible=False), outputs=image_modal)
    open_modal_btn.click(lambda: gr.update(visible=True), outputs=image_modal)

    with gr.Tabs() as tabs:

        # ── Tab 1: Image Upload ───────────────────────────────────────────────
        with gr.Tab("1. 🌸 Image Upload"):
            gr.HTML("<div class='screen-title'>🌸 Plant Image Analysis</div>")
            gr.HTML("<div style='background:#f0fff4;border:1px solid #a8d5a2;border-radius:8px;padding:8px 14px;font-size:13px;color:#2C5F2D;margin-bottom:8px;'>📌 <b>How to use:</b> Choose your plant type, upload a photo, then click <b>Analyze plant</b> to get an instant health report.</div>")
            gr.Markdown("Supports upload, webcam, and clipboard paste.")
            with gr.Row():
                with gr.Column():
                    plant_type    = gr.Dropdown(choices=['Phalaenopsis','Cymbidium','Cremastra','Other'], value='Phalaenopsis', label="Plant type")
                    image_input   = gr.Image(type="pil", label="Plant image", sources=["upload","webcam","clipboard"])
                    analyze_btn   = gr.Button("Analyze plant", variant="primary")
                with gr.Column():
                    result_output         = gr.Markdown(label="Analysis result")
                    session_summary_output = gr.Textbox(label="Session Summary", interactive=False)

            def ui_analyze_wrapper(img, p_type):
                if img is None:
                    return "Please upload an image before analysis.", ""
                return analyze_plant(img, p_type)

            analyze_btn.click(fn=ui_analyze_wrapper, inputs=[image_input, plant_type], outputs=[result_output, session_summary_output])

        # ── Tab 2: IoT Sensors ────────────────────────────────────────────────
        with gr.Tab("2. 📡 IoT Sensors"):
            gr.HTML("<div class='screen-title'>📡 Real-Time Sensor Data</div>")
            gr.HTML("<div style='background:#f0fff4;border:1px solid #a8d5a2;border-radius:8px;padding:8px 14px;font-size:13px;color:#2C5F2D;margin-bottom:8px;'>📌 <b>How to use:</b> Click <b>Sample now</b> to fetch live sensor data. Repeat to build a trend graph.</div>")
            gr.HTML(
                "<div style='background:#fff3cd;border:1px solid #ffc107;border-radius:8px;"
                "padding:8px 14px;font-size:13px;color:#856404;margin-bottom:8px;'>"
                "⏳ <b>Note:</b> The IoT server may take up to 30 seconds to wake up on first use.</div>"
            )
            with gr.Row():
                sample_btn = gr.Button("Sample now", variant="primary")
                clear_btn = gr.Button("Clear graph")
            sensor_status = gr.Textbox(label="Current status", lines=8)
            sensor_plot   = gr.Plot(label="Sensor Trend")
            sample_btn.click(sample_sensors,       outputs=[sensor_status, sensor_plot])
            clear_btn.click(clear_sensor_display, outputs=[sensor_status, sensor_plot])

        # ── Tab 3: Article Search ─────────────────────────────────────────────
        with gr.Tab("3. 🔍 Article Search"):
            gr.HTML("<div class='screen-title'>🔍 Article Search Engine (RAG)</div>")
            gr.Markdown(
                "Ask any question about orchid diseases. The answer is extracted from the 5 academic articles using semantic search (RAG).\n\n"
                "**Click an example or type your own question:**"
            )
            with gr.Row():
                ex1 = gr.Button("🌿 What causes root rot?",             size="sm")
                ex2 = gr.Button("🍂 How to treat leaf blight?",          size="sm")
                ex3 = gr.Button("🌡️ Optimal temperature for orchids?",  size="sm")
                ex4 = gr.Button("🍄 What is Fusarium wilt?",             size="sm")
            with gr.Row():
                query_input = gr.Textbox(label="Your question", placeholder="e.g. What causes leaf blight?", scale=4)
                search_btn  = gr.Button("Search", variant="primary", scale=1)
            results_output = gr.Markdown()
            search_btn.click(run_search,  inputs=query_input, outputs=results_output)
            query_input.submit(run_search, inputs=query_input, outputs=results_output)
            ex1.click(lambda: "What causes root rot?",                     outputs=query_input)
            ex2.click(lambda: "How to treat leaf blight?",                 outputs=query_input)
            ex3.click(lambda: "What is the optimal temperature for orchids?", outputs=query_input)
            ex4.click(lambda: "What is Fusarium wilt?",                    outputs=query_input)

        # ── Tab 4: Dashboard ──────────────────────────────────────────────────
        with gr.Tab("4. 📊 Dashboard") as tab_dashboard:
            gr.HTML("<div class='screen-title'>📊 Plant Status Dashboard</div>")
            gr.Markdown("Live overview of sensor readings and image history. Auto-refreshes every 30 seconds. All times in Israel timezone.")
            with gr.Row():
                date_filter = gr.Dropdown(
                    choices=[("All time", 0), ("Today", 1), ("Last 7 days", 7), ("Last 30 days", 30)],
                    value=0, label="📅 Quick filter"
                )
                export_btn = gr.Button("📥 Export CSV", variant="secondary")

            def refresh_with_filters(days_filter=0):
                filtered = filter_sensor_history(days_filter)
                if days_filter == 0:
                    f_images = uploaded_images
                else:
                    now = datetime.datetime.now(ISRAEL_TZ)
                    cutoff = (now if days_filter == 1 else now - datetime.timedelta(days=int(days_filter) - 1)).replace(
                        hour=0, minute=0, second=0, microsecond=0)
                    cutoff_str = cutoff.strftime('%Y-%m-%d %H:%M:%S')
                    f_images = [img for img in uploaded_images if img.get('time', '') >= cutoff_str]
                return get_overview(filtered), get_dashboard_plot(filtered, f_images), get_images_table(), generate_care_report()

            export_file = gr.File(label="Download CSV", visible=False)
            overview_md = gr.Markdown(value=get_overview())
            dash_plot   = gr.Plot()
            images_df   = gr.Dataframe(value=get_images_table(), label="Image upload history")
            report_md   = gr.Markdown(value=generate_care_report())

            date_filter.change(refresh_with_filters, inputs=[date_filter], outputs=[overview_md, dash_plot, images_df, report_md])
            auto_timer = gr.Timer(value=30)
            auto_timer.tick(refresh_with_filters, inputs=[date_filter], outputs=[overview_md, dash_plot, images_df, report_md])

            def export_csv():
                import tempfile, csv
                if not sensor_history:
                    return gr.update(visible=False)
                tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.csv', mode='w', newline='')
                writer = csv.DictWriter(tmp, fieldnames=['time','temperature','humidity','soil_moisture','light'])
                writer.writeheader()
                for row in sensor_history:
                    writer.writerow({k: row.get(k, '') for k in ['time','temperature','humidity','soil_moisture','light']})
                tmp.close()
                return gr.update(visible=True, value=tmp.name)

            export_btn.click(export_csv, outputs=[export_file])

        # ── Tab 5: Temperature Advisor ────────────────────────────────────────
        with gr.Tab("5. 🌡️ Temperature Advisor") as tab_temp:
            gr.HTML("<div class='screen-title'>🌡️ Room Temperature Care Advisor</div>")
            gr.Markdown(
                "Reads **live room temperature** from the IoT sensor and compares it to "
                "optimal ranges from the 5 academic articles. Updates automatically every 60 seconds."
            )
            gr.HTML("<div style='background:#f0fff4;border:1px solid #a8d5a2;border-radius:8px;padding:8px 14px;font-size:13px;color:#2C5F2D;margin-bottom:8px;'>📌 Live temperature is fetched automatically. Data refreshes every 60 seconds.</div>")
            with gr.Row():
                with gr.Column(scale=1):
                    s5_gauge = gr.Plot(label="Temperature Gauge")
                with gr.Column(scale=1):
                    s5_advice = gr.Markdown(value="Loading live temperature...")
            temp_timer = gr.Timer(value=60)
            temp_timer.tick(temperature_advisor, outputs=[s5_advice, s5_gauge])

        # ── Tab 6: AI Assistant ───────────────────────────────────────────────
        with gr.Tab("6. 🤖 AI Assistant"):
            gr.HTML("<div class='screen-title'>🤖 AgriCloud AI Assistant</div>")
            gr.HTML("""
                <div style="background:linear-gradient(135deg,#f0fff4,#e8f8f0);
                    border:1.5px solid #a8d5a2;border-radius:12px;padding:14px 20px;margin-bottom:14px;
                    display:flex;align-items:center;gap:16px;flex-wrap:wrap">
                    <div style="font-size:36px">🌿</div>
                    <div>
                        <div style="font-size:16px;font-weight:bold;color:#2C5F2D">AgriCloud AI Assistant</div>
                        <div style="font-size:13px;color:#555;margin-top:4px">
                            Ask me about orchid diseases, care tips, and sensor readings.<br>
                            My answers are based on <b style='color:#2C5F2D;'>5 academic articles</b> in the database.
                        </div>
                    </div>
                    <div style="margin-left:auto;display:flex;gap:8px;flex-wrap:wrap">
                        <span style="background:#e8f5e9;border:1px solid #a5d6a7;border-radius:20px;padding:4px 12px;font-size:12px;color:#2C5F2D">🦠 Diseases</span>
                        <span style="background:#e3f2fd;border:1px solid #90caf9;border-radius:20px;padding:4px 12px;font-size:12px;color:#1565c0">🌡️ Temperature</span>
                        <span style="background:#fff8e1;border:1px solid #ffe082;border-radius:20px;padding:4px 12px;font-size:12px;color:#e65100">💧 Watering</span>
                        <span style="background:#fce4ec;border:1px solid #f48fb1;border-radius:20px;padding:4px 12px;font-size:12px;color:#880e4f">🌸 Care tips</span>
                    </div>
                </div>
            """)
            chatbot_display = gr.Chatbot(
                label="Conversation", height=420, show_label=False,
                avatar_images=(None, "https://em-content.zobj.net/source/google/387/seedling_1f331.png")
            )
            with gr.Row():
                chat_msg   = gr.Textbox(placeholder="Ask about your orchid...", label="", show_label=False, scale=5, lines=1)
                chat_send  = gr.Button("Send 💬", variant="primary", scale=1, min_width=100)
                chat_clear = gr.Button("🗑️", scale=0, min_width=50)

            def chatbot_with_count(msg, history):
                """Streams chatbot response and saves a timestamp entry to Firebase chat_history."""
                if not msg or not msg.strip():
                    yield from chatbot_respond_stream(msg, history)
                    return
                for chunk in chatbot_respond_stream(msg, history):
                    yield chunk
                # Save to Firebase for points tracking (replaces chat_count global)
                try:
                    import requests as _req
                    ts  = datetime.datetime.now(ISRAEL_TZ).strftime('%Y-%m-%d %H:%M:%S')
                    key = ts.replace(':', '-').replace(' ', '_')
                    _req.put(
                        f'https://orchid-shark-default-rtdb.firebaseio.com/chat_history/{key}.json',
                        json={'time': ts, 'pts': 10}, timeout=3
                    )
                except:
                    pass

            chat_send.click(chatbot_with_count,  inputs=[chat_msg, chatbot_display], outputs=[chatbot_display, chat_msg])
            chat_msg.submit(chatbot_with_count,  inputs=[chat_msg, chatbot_display], outputs=[chatbot_display, chat_msg])
            chat_clear.click(lambda: ([], ""),   outputs=[chatbot_display, chat_msg])

        # ── Tab 7: Achievements ───────────────────────────────────────────────
        with gr.Tab("7. 🏆 Achievements") as tab_achievements:
            gr.HTML("<div class='screen-title'>🏆 Achievements & Progress</div>")
            with gr.Row():
                ach_view = gr.Dropdown(
                    choices=[("Today", "today"), ("Last 7 days", "week"), ("All time", "alltime")],
                    value="today", label="📊 View"
                )
            achievements_html = gr.HTML(value="<p style='color:#888'>Loading...</p>")

            def save_daily_score_to_firebase(date_str, points):
                """Saves today's computed point total to Firebase daily_scores/<date>."""
                try:
                    import requests as _req
                    _req.put(
                        f'https://orchid-shark-default-rtdb.firebaseio.com/daily_scores/{date_str}.json',
                        json=points, timeout=5
                    )
                except Exception as e:
                    print(f"Firebase daily score save error: {e}")

            def load_daily_scores_from_firebase(days=7):
                """Loads daily score entries from Firebase for the last N days."""
                try:
                    import requests as _req
                    r    = _req.get('https://orchid-shark-default-rtdb.firebaseio.com/daily_scores.json', timeout=5)
                    data = r.json()
                    if not data or not isinstance(data, dict):
                        return {}
                    cutoff = (datetime.datetime.now(ISRAEL_TZ) - datetime.timedelta(days=days)).strftime('%Y-%m-%d')
                    return {k: v for k, v in data.items() if k >= cutoff}
                except Exception as e:
                    print(f"Firebase daily score load error: {e}")
                    return {}

            def compute_daily_points(date_str):
                """
                Computes total points for a given date by counting:
                - sensor samples (×10) and images (×25) from in-memory lists
                - chats (×10) and searches (×5) from Firebase chat_history / search_history
                """
                day_samples = sum(1 for r in sensor_history    if r.get('time', '').startswith(date_str))
                day_images  = sum(1 for img in uploaded_images if img.get('time', '').startswith(date_str))
                try:
                    import requests as _rq
                    ch  = _rq.get('https://orchid-shark-default-rtdb.firebaseio.com/chat_history.json',   timeout=3).json()
                    sh  = _rq.get('https://orchid-shark-default-rtdb.firebaseio.com/search_history.json', timeout=3).json()
                    day_chats   = sum(1 for v in ch.values() if v.get('time','').startswith(date_str)) if ch and isinstance(ch, dict) else 0
                    day_searches= sum(1 for v in sh.values() if v.get('time','').startswith(date_str)) if sh and isinstance(sh, dict) else 0
                except:
                    day_chats = day_searches = 0
                return day_samples * 10 + day_images * 25 + day_chats * 10 + day_searches * 5

            def get_achievements(view="today"):
                """
                Builds the achievements HTML for the selected view (today / week / alltime).
                All activity counts (chats, searches) are read from Firebase — no local counters.
                """
                today_str = datetime.datetime.now(ISRAEL_TZ).strftime('%Y-%m-%d')
                samples   = len(sensor_history)
                images    = len(uploaded_images)

                try:
                    import requests as _rq
                    ch  = _rq.get('https://orchid-shark-default-rtdb.firebaseio.com/chat_history.json',   timeout=3).json()
                    sh  = _rq.get('https://orchid-shark-default-rtdb.firebaseio.com/search_history.json', timeout=3).json()
                    chats          = len(ch) if ch and isinstance(ch, dict) else 0
                    searches       = len(sh) if sh and isinstance(sh, dict) else 0
                    today_chats    = sum(1 for v in ch.values() if v.get('time','').startswith(today_str)) if ch and isinstance(ch, dict) else 0
                    today_searches = sum(1 for v in sh.values() if v.get('time','').startswith(today_str)) if sh and isinstance(sh, dict) else 0
                except:
                    chats = searches = today_chats = today_searches = 0

                alltime_points = samples * 10 + images * 25 + searches * 5 + chats * 10
                today_samples  = sum(1 for r in sensor_history    if r.get('time', '').startswith(today_str))
                today_images   = sum(1 for img in uploaded_images if img.get('time', '').startswith(today_str))
                today_points   = compute_daily_points(today_str)
                save_daily_score_to_firebase(today_str, today_points)

                if view == "today":
                    if   today_points >= 500: level, level_num, next_pts, level_color = "Master Grower", 4, 500, "#8e44ad"
                    elif today_points >= 250: level, level_num, next_pts, level_color = "Bloomer",       3, 500, "#e67e22"
                    elif today_points >= 100: level, level_num, next_pts, level_color = "Sprout",        2, 250, "#27ae60"
                    else:                     level, level_num, next_pts, level_color = "Seedling",      1, 100, "#3498db"

                    progress_pct = min(100, int(today_points / next_pts * 100))
                    icon = {1:"🌱", 2:"🌿", 3:"🌺", 4:"🏆"}.get(level_num, "🌱")

                    badges = []
                    if samples >= 1:  badges.append(("🌡️", "First Sample",       "Take your first sensor reading"))
                    if images  >= 1:  badges.append(("📸", "Plant Photographer", "Upload your first plant photo"))
                    if searches>= 1:  badges.append(("🔍", "Disease Investigator","Search the article database"))
                    if chats   >= 1:  badges.append(("🤖", "AI Curious",         "Ask the AI assistant"))
                    if samples >= 10: badges.append(("📊", "Sensor Pro",          "Take 10 sensor readings"))
                    if samples >= 1 and images >= 1 and searches >= 1 and chats >= 1:
                        badges.append(("🌸", "Devoted Grower", "Use all 4 main features"))
                    _, pct = stability_service.get_stability(sensor_history)
                    if pct is not None and pct == 100:
                        badges.append(("✅", "Perfect Environment", "100% readings within normal range"))

                    badges_html = "".join(f"""
                        <div style="display:inline-flex;align-items:center;gap:10px;
                            background:linear-gradient(135deg,#f0fff4,#e8f8e8);
                            border:1.5px solid #27ae60;border-radius:12px;
                            padding:10px 16px;margin:6px;min-width:200px;">
                            <span style="font-size:28px">{e}</span>
                            <div><div style="font-weight:bold;color:#2C5F2D;font-size:14px">{n}</div>
                            <div style="color:#666;font-size:11px">{d}</div></div>
                        </div>""" for e, n, d in badges)

                    return f"""
                    <style>
                      @keyframes crown-bounce {{0%,100%{{transform:translateY(0) rotate(-5deg)}}50%{{transform:translateY(-8px) rotate(5deg)}}}}
                      @keyframes pts-pop {{0%{{transform:scale(0.5);opacity:0}}70%{{transform:scale(1.15)}}100%{{transform:scale(1);opacity:1}}}}
                      @keyframes sparkle {{0%,100%{{opacity:0;transform:scale(0)}}50%{{opacity:1;transform:scale(1)}}}}
                      .crown{{animation:crown-bounce 2s ease-in-out infinite;display:inline-block}}
                      .pts-num{{animation:pts-pop 0.6s cubic-bezier(.36,.07,.19,.97)}}
                      .spark{{animation:sparkle 1.5s ease-in-out infinite}}
                      .spark:nth-child(2){{animation-delay:0.3s}}.spark:nth-child(3){{animation-delay:0.6s}}
                    </style>
                    <div style="font-family:'Carlito',Calibri,sans-serif;padding:10px">
                      <div style="text-align:center;margin-bottom:24px;position:relative">
                        {"<div class='crown' style='font-size:56px;margin-bottom:-10px'>👑</div>" if level_num >= 3 else ""}
                        <span class="spark" style="font-size:24px;position:absolute;top:0;left:30%">✨</span>
                        <span class="spark" style="font-size:20px;position:absolute;top:10px;right:28%">⭐</span>
                        <span class="spark" style="font-size:18px;position:absolute;top:5px;right:35%">💫</span>
                        <div class="pts-num" style="font-size:80px;font-weight:bold;color:{level_color};line-height:1;text-shadow:0 4px 16px {level_color}55">{today_points}</div>
                        <div style="font-size:18px;color:#666;margin-top:4px;letter-spacing:2px;text-transform:uppercase">Today's Points</div>
                        <div style="font-size:14px;color:{level_color};margin-top:4px">All-Time: {alltime_points} pts</div>
                      </div>
                      <div style="text-align:center;margin-bottom:20px">
                        <div style="display:inline-block;background:linear-gradient(135deg,{level_color},{level_color}cc);color:white;
                            border-radius:30px;padding:12px 32px;font-size:20px;font-weight:bold;
                            box-shadow:0 6px 20px {level_color}55;letter-spacing:1px">
                            {icon} Level {level_num} — {level}
                        </div>
                      </div>
                      <div style="margin:0 auto 24px auto;max-width:500px">
                        <div style="display:flex;justify-content:space-between;font-size:12px;color:#888;margin-bottom:4px">
                            <span>Progress to next level</span><span>{today_points} / {next_pts} pts</span>
                        </div>
                        <div style="background:#e0e0e0;border-radius:20px;height:20px;overflow:hidden;box-shadow:inset 0 2px 4px rgba(0,0,0,0.1)">
                            <div style="width:{progress_pct}%;height:100%;background:linear-gradient(90deg,{level_color},{level_color}99);
                                border-radius:20px;display:flex;align-items:center;justify-content:center;
                                color:white;font-size:11px;font-weight:bold">{progress_pct}%</div>
                        </div>
                      </div>
                      <div style="display:flex;justify-content:center;gap:16px;flex-wrap:wrap;margin-bottom:24px">
                        <div style="text-align:center;background:#fff3cd;border-radius:12px;padding:12px 20px;min-width:100px">
                            <div style="font-size:28px;font-weight:bold;color:#e67e22">{today_samples}</div>
                            <div style="font-size:11px;color:#666">Sensor Samples<br><b>×10 pts</b></div>
                        </div>
                        <div style="text-align:center;background:#d4edda;border-radius:12px;padding:12px 20px;min-width:100px">
                            <div style="font-size:28px;font-weight:bold;color:#27ae60">{today_images}</div>
                            <div style="font-size:11px;color:#666">Images Analyzed<br><b>×25 pts</b></div>
                        </div>
                        <div style="text-align:center;background:#cce5ff;border-radius:12px;padding:12px 20px;min-width:100px">
                            <div style="font-size:28px;font-weight:bold;color:#2980b9">{today_searches}</div>
                            <div style="font-size:11px;color:#666">RAG Searches<br><b>×5 pts</b></div>
                        </div>
                        <div style="text-align:center;background:#e8d5f5;border-radius:12px;padding:12px 20px;min-width:100px">
                            <div style="font-size:28px;font-weight:bold;color:#8e44ad">{today_chats}</div>
                            <div style="font-size:11px;color:#666">AI Conversations<br><b>×10 pts each</b></div>
                        </div>
                      </div>
                      <div style="background:#f8f9fa;border:1.5px solid #dee2e6;border-radius:12px;padding:16px 20px;margin-bottom:20px">
                        <div style="font-size:15px;font-weight:bold;color:#2C5F2D;margin-bottom:12px">📋 How to earn points</div>
                        <div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;font-size:13px">
                            <div style="display:flex;align-items:center;gap:8px;padding:6px 10px;background:white;border-radius:8px;border:1px solid #eee">
                                <span style="font-size:20px">🌡️</span>
                                <div><b>Sample sensors</b><br><span style="color:#e67e22">+10 pts</span> per click</div>
                            </div>
                            <div style="display:flex;align-items:center;gap:8px;padding:6px 10px;background:white;border-radius:8px;border:1px solid #eee">
                                <span style="font-size:20px">📸</span>
                                <div><b>Upload & analyze image</b><br><span style="color:#27ae60">+25 pts</span> per photo</div>
                            </div>
                            <div style="display:flex;align-items:center;gap:8px;padding:6px 10px;background:white;border-radius:8px;border:1px solid #eee">
                                <span style="font-size:20px">🔍</span>
                                <div><b>Search articles</b><br><span style="color:#2980b9">+5 pts</span> per search</div>
                            </div>
                            <div style="display:flex;align-items:center;gap:8px;padding:6px 10px;background:white;border-radius:8px;border:1px solid #eee">
                                <span style="font-size:20px">🤖</span>
                                <div><b>Chat with AI</b><br><span style="color:#8e44ad">+10 pts</span> per message</div>
                            </div>
                            <div style="display:flex;align-items:center;gap:8px;padding:6px 10px;background:white;border-radius:8px;border:1px solid #eee">
                                <span style="font-size:20px">🕵️</span>
                                <div><b>Use Care Agent</b><br><span style="color:#16a085">+10 pts</span> per diagnosis</div>
                            </div>
                            <div style="display:flex;align-items:center;gap:8px;padding:6px 10px;background:linear-gradient(135deg,#fff8e1,#fff3cd);border-radius:8px;border:1px solid #ffe082">
                                <span style="font-size:20px">⭐</span>
                                <div><b>100 pts</b> → Sprout &nbsp;|&nbsp; <b>250</b> → Bloomer &nbsp;|&nbsp; <b>500</b> → Master</div>
                            </div>
                        </div>
                      </div>
                      <div style="margin-bottom:16px">
                        <div style="font-size:16px;font-weight:bold;color:#2C5F2D;margin-bottom:10px">🏅 Badges Unlocked ({len(badges)})</div>
                        <div style="display:flex;flex-wrap:wrap">{badges_html or "<span style='color:#888;font-style:italic'>No badges yet!</span>"}</div>
                      </div>
                    </div>"""

                elif view in ("history", "week"):
                    scores      = load_daily_scores_from_firebase(7)
                    scores[today_str] = today_points
                    if not scores:
                        return "<p style='color:#888;padding:20px'>No history yet. Use the app to earn points!</p>"
                    sorted_days  = sorted(scores.keys(), reverse=True)
                    weekly_total = sum(scores.values())
                    best_day     = max(scores, key=scores.get)
                    max_pts      = max(scores.values()) or 1
                    bars = ""
                    for day in sorted_days:
                        pts      = scores[day]
                        pct      = int(pts / max_pts * 100)
                        is_today = day == today_str
                        color    = "#27ae60" if is_today else "#3498db"
                        label    = "Today" if is_today else day
                        bars += f"""
                        <div style="margin-bottom:12px">
                            <div style="display:flex;justify-content:space-between;font-size:13px;margin-bottom:4px">
                                <span style="color:{'#27ae60' if is_today else '#333'};font-weight:{'bold' if is_today else 'normal'}">{label}</span>
                                <span style="font-weight:bold;color:{color}">{pts} pts</span>
                            </div>
                            <div style="background:#e0e0e0;border-radius:10px;height:16px;overflow:hidden">
                                <div style="width:{pct}%;height:100%;background:linear-gradient(90deg,{color},{color}99);border-radius:10px"></div>
                            </div>
                        </div>"""
                    return f"""
                    <div style="font-family:'Carlito',Calibri,sans-serif;padding:16px">
                      <div style="display:flex;gap:20px;flex-wrap:wrap;margin-bottom:24px;justify-content:center">
                        <div style="text-align:center;background:linear-gradient(135deg,#f0fff4,#e8f8e8);border:2px solid #27ae60;border-radius:16px;padding:16px 28px">
                            <div style="font-size:40px;font-weight:bold;color:#27ae60">{weekly_total}</div>
                            <div style="color:#666;font-size:13px">Weekly Total</div>
                        </div>
                        <div style="text-align:center;background:linear-gradient(135deg,#fff8e1,#fff3cd);border:2px solid #e67e22;border-radius:16px;padding:16px 28px">
                            <div style="font-size:28px;font-weight:bold;color:#e67e22">⭐ {scores.get(best_day,0)}</div>
                            <div style="color:#666;font-size:13px">Best Day<br>{best_day}</div>
                        </div>
                        <div style="text-align:center;background:linear-gradient(135deg,#e8f4fd,#cce5ff);border:2px solid #3498db;border-radius:16px;padding:16px 28px">
                            <div style="font-size:28px;font-weight:bold;color:#3498db">{round(weekly_total/max(len(scores),1))}</div>
                            <div style="color:#666;font-size:13px">Daily Average</div>
                        </div>
                      </div>
                      <div style="max-width:600px;margin:0 auto">
                        <div style="font-size:16px;font-weight:bold;color:#2C5F2D;margin-bottom:14px">📅 Daily Breakdown</div>
                        {bars}
                      </div>
                    </div>"""

                else:  # alltime
                    scores       = load_daily_scores_from_firebase(365)
                    scores[today_str] = today_points
                    total        = sum(scores.values())
                    days_active  = len([v for v in scores.values() if v > 0])
                    return f"""
                    <div style="font-family:'Carlito',Calibri,sans-serif;padding:16px;text-align:center">
                        <div style="font-size:64px;font-weight:bold;color:#8e44ad">{total}</div>
                        <div style="font-size:18px;color:#666;margin-bottom:20px">All-Time Points</div>
                        <div style="display:flex;gap:20px;justify-content:center;flex-wrap:wrap">
                            <div style="background:#e8d5f5;border-radius:12px;padding:14px 24px">
                                <div style="font-size:28px;font-weight:bold;color:#8e44ad">{days_active}</div>
                                <div style="font-size:12px;color:#666">Active Days</div>
                            </div>
                            <div style="background:#d4edda;border-radius:12px;padding:14px 24px">
                                <div style="font-size:28px;font-weight:bold;color:#27ae60">{round(total/max(days_active,1))}</div>
                                <div style="font-size:12px;color:#666">Avg pts/day</div>
                            </div>
                            <div style="background:#fff3cd;border-radius:12px;padding:14px 24px">
                                <div style="font-size:28px;font-weight:bold;color:#e67e22">{alltime_points}</div>
                                <div style="font-size:12px;color:#666">This Session</div>
                            </div>
                        </div>
                    </div>"""

            ach_view.change(get_achievements, inputs=[ach_view], outputs=[achievements_html])

        # ── Tab 8: Care Agent ─────────────────────────────────────────────────
        with gr.Tab("8. 🕵️ Care Agent"):
            gr.HTML("<div class='screen-title'>🕵️ DiagnosisAgent - Smart Care Assistant</div>")
            gr.HTML("""
                <div style="background:linear-gradient(135deg,#f0fff4,#e8f8f0);
                    border:1.5px solid #a8d5a2;border-radius:12px;padding:16px 20px;margin-bottom:16px;">
                    <div style="font-size:15px;font-weight:bold;color:#2C5F2D;margin-bottom:8px">🧠 How the agent works</div>
                    <div style="display:flex;gap:12px;flex-wrap:wrap;font-size:13px;color:#444;">
                        <div style="background:white;color:#444;border-radius:8px;padding:8px 14px;border:1px solid #d4edda">🌡️ Detects <b>temperature</b> keywords → calls IoT server</div>
                        <div style="background:white;color:#444;border-radius:8px;padding:8px 14px;border:1px solid #d4edda">💧 Detects <b>watering</b> keywords → calls WateringScheduleService</div>
                        <div style="background:white;color:#444;border-radius:8px;padding:8px 14px;border:1px solid #d4edda">🔬 Detects <b>disease</b> keywords → searches academic articles (RAG)</div>
                        <div style="background:white;color:#444;border-radius:8px;padding:8px 14px;border:1px solid #d4edda">📊 Detects <b>environment</b> keywords → calls EnvironmentStabilityService</div>
                    </div>
                </div>
            """)
            gr.HTML("""
                <div style="background:#fff8e1;border:1px solid #ffe082;border-radius:8px;
                    padding:10px 16px;margin-bottom:12px;font-size:13px;color:#795548">
                    💡 <b style='color:#a36d02;'>Try:</b> "My leaves are yellowing and it feels warm" &nbsp;|&nbsp;
                    "Should I water now?" &nbsp;|&nbsp; "Check the overall environment status"
                </div>
            """)
            with gr.Row():
                agent_input = gr.Textbox(
                    label="Describe your orchid's condition",
                    placeholder="e.g. The leaves are turning yellow and the temperature feels high...",
                    lines=1, max_lines=1, scale=4
                )
                agent_btn = gr.Button("🔍 Run Agent", variant="primary", scale=1, min_width=120)
            with gr.Row():
                with gr.Column(scale=1):
                    gr.HTML("<div style='font-weight:bold;color:#2C5F2D;margin-bottom:4px'>🧩 Agent reasoning</div>")
                    agent_steps  = gr.Markdown(value="_Agent steps will appear here..._")
                with gr.Column(scale=1):
                    gr.HTML("<div style='font-weight:bold;color:#2C5F2D;margin-bottom:4px'>🌸 Diagnosis Report</div>")
                    agent_output = gr.Markdown(value="_Diagnosis will appear here..._")

            def run_diagnosis_agent(user_input):
                """
                Keyword-based agent that routes the user's description to the
                appropriate tool(s): IoT sensor, WateringScheduleService,
                RAG article search, or EnvironmentStabilityService.
                Falls back to running all tools if no keywords are matched.
                """
                if not user_input or not user_input.strip():
                    return "", "Please describe your orchid's condition."

                text    = user_input.lower()
                steps   = ["## 🕵️ DiagnosisAgent - Reasoning\n"]
                results = []

                # ── Tool 1: IoT / Temperature ─────────────────────────────────
                temp_keywords = ["temperature","warm","hot","cold","heat","cool","temp","חם","קר","טמפרטורה"]
                if any(kw in text for kw in temp_keywords):
                    steps.append("**🌡️ Tool called: IoT Temperature Sensor**")
                    steps.append("→ Reason: temperature-related keywords detected")
                    temp = get_iot_values(feed="temperature", single=True)
                    if temp:
                        p = ORCHID_TEMP_PROFILE
                        if   temp >= p['danger_high']: label = "🔴 DANGER - Too Hot"
                        elif temp >= p['stress_high']: label = "🟠 Warning - Too Warm"
                        elif temp >= p['ideal_min']:   label = "✅ Optimal"
                        else:                          label = "🔵 Too Cool"
                        steps.append(f"→ Result: {temp}°C — {label}\n")
                        results.append(f"**Temperature:** {temp}°C — {label}")
                    else:
                        steps.append("→ Result: IoT server unavailable\n")

                # ── Tool 2: Watering ──────────────────────────────────────────
                water_keywords = ["water","watering","soil","moisture","dry","wet","irrigat","השקי","קרקע","יבש"]
                if any(kw in text for kw in water_keywords):
                    steps.append("**💧 Tool called: WateringScheduleService**")
                    steps.append("→ Reason: watering/soil-related keywords detected")
                    rec_text, urgency = watering_service.get_recommendation(sensor_history)
                    icon = {'urgent':'🔴','watch':'🟡','ok':'🟢','unknown':'⚪'}.get(urgency,'⚪')
                    steps.append(f"→ Result: {icon} {rec_text}\n")
                    tip = ""
                    if urgency == "urgent": tip = "\n\n💡 **Tip:** Water thoroughly until it drains from the bottom, then wait for the soil to partially dry."
                    elif urgency == "watch": tip = "\n\n💡 **Tip:** Check the soil with your finger - if the top 2cm feel dry, it's time to water."
                    elif urgency == "ok":   tip = "\n\n💡 **Tip:** Orchids prefer to dry out slightly between waterings - avoid overwatering."
                    results.append(f"**Watering:** {icon} {rec_text}{tip}")

                # ── Tool 3: Disease / RAG ─────────────────────────────────────
                disease_keywords = ["yellow","brown","spot","rot","disease","sick","leaf","blight","virus","fungal","wilt","צהוב","חום","מחלה","כתם","עלה"]
                if any(kw in text for kw in disease_keywords):
                    steps.append("**🔬 Tool called: RAG Article Search**")
                    steps.append("→ Reason: disease/symptom keywords detected")
                    # Sanity check before calling RAG (requires at least 2 words)
                    if len(user_input.strip().split()) >= 2:
                        search_result = run_rag_query(user_input, top_k=2)
                        if search_result:
                            snippet = search_result[:300].replace('\n', ' ')
                            steps.append("→ Result: found relevant articles\n")
                            results.append(f"**Disease info (from academic articles):**\n{snippet}...")
                        else:
                            steps.append("→ Result: no articles found\n")
                    else:
                        steps.append("→ Skipped: query too short for RAG search\n")

                # ── Tool 4: Environment stability ─────────────────────────────
                env_keywords = ["environment","stable","stability","conditions","overall","general","status","סביבה","יציבות","מצב"]
                if any(kw in text for kw in env_keywords):
                    steps.append("**📊 Tool called: EnvironmentStabilityService**")
                    steps.append("→ Reason: environment/stability keywords detected")
                    stability_text, pct = stability_service.get_stability(sensor_history)
                    steps.append(f"→ Result: {stability_text}\n")
                    if pct is None:        interp = "No sensor data yet - take readings first."
                    elif pct == 100:       interp = "✅ Perfect - all readings within normal range."
                    elif pct >= 70:        interp = f"🟡 Mostly stable ({pct}%) - occasional issues detected."
                    elif pct >= 40:        interp = f"🟠 Unstable ({pct}%) - frequent threshold violations."
                    else:                  interp = f"🔴 Critical ({pct}%) - most readings are out of range."
                    results.append(f"**Environment stability:** {stability_text}\n\n{interp}")

                # ── Fallback: no keywords matched ─────────────────────────────
                if not results:
                    steps.append("**⚠️ No specific keywords detected** - running all tools\n")
                    temp = get_iot_values(feed="temperature", single=True)
                    if temp:
                        results.append(f"**Temperature:** {temp}°C")
                    rec_text, urgency = watering_service.get_recommendation(sensor_history)
                    icon = {'urgent':'🔴','watch':'🟡','ok':'🟢','unknown':'⚪'}.get(urgency,'⚪')
                    results.append(f"**Watering:** {icon} {rec_text}")
                    stability_text, _ = stability_service.get_stability(sensor_history)
                    results.append(f"**Environment stability:** {stability_text}")

                return "\n".join(steps), "## 🌸 Agent Diagnosis Report\n\n" + "\n\n".join(results)

            agent_btn.click(run_diagnosis_agent,   inputs=[agent_input], outputs=[agent_steps, agent_output])
            agent_input.submit(run_diagnosis_agent, inputs=[agent_input], outputs=[agent_steps, agent_output])

    # ── Auto-refresh on tab select ────────────────────────────────────────────
    tab_dashboard.select(refresh_with_filters,  inputs=[date_filter], outputs=[overview_md, dash_plot, images_df, report_md])
    tab_temp.select(temperature_advisor,         outputs=[s5_advice, s5_gauge])
    tab_achievements.select(lambda v: get_achievements(v), inputs=[ach_view], outputs=[achievements_html])

    # Load temperature on app start
    app.load(temperature_advisor, outputs=[s5_advice, s5_gauge])

app.launch(share=True, show_api=False)

/tmp/ipykernel_2498/1867219497.py:36: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2498/1867219497.py:36: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2498/1867219497.py:214: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_display = gr.Chatbot(
/tmp/ipykernel_2498/1867219497.py:214: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=Fa

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0df1a11ad7b3157129.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
